Handles millions of documents
Uses GPU acceleration
Prevents data loss (checkpointing)
Keeps vector + metadata synced
Memory efficient (streaming + batching)

In [1]:
!pip install faiss-cpu sentence-transformers

import os
import glob
import faiss
import sqlite3
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

In [2]:
from google.colab import drive
drive.mount('/content/drive')


# ==============================
# 2. PATHS
# ==============================
CSV_1 = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/1.2df_txt_norm.csv"
CSV_2 = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/2.2df_CSV_norm.csv"
PARQUET_DIR = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/3.2_clean_chunks"

FAISS_PATH = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/4.faiss_index.index"
SQLITE_PATH = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/4.faiss_metadata.db"
CHECKPOINT_PATH = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/4.faiss_checkpoint.txt"

Mounted at /content/drive


In [ ]:
# import os

# os.remove(FAISS_PATH)
# os.remove(SQLITE_PATH)
# os.remove(CHECKPOINT_PATH)

In [ ]:
# import os

# files = [
#     FAISS_PATH,
#     SQLITE_PATH,
#     CHECKPOINT_PATH
# ]

# for f in files:
#     if os.path.exists(f):
#         os.remove(f)
#         print("Deleted:", f)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

print("Model loaded on:", device)

In [4]:
def stream_all_data(start_doc_id=0):
    current_id = 0

    # CSV 1
    for chunk in pd.read_csv(CSV_1, chunksize=5000):
        for t in chunk["text"].dropna().astype(str):
            if current_id >= start_doc_id:
                yield t, "CSV_1"
            current_id += 1

    # CSV 2
    for chunk in pd.read_csv(CSV_2, chunksize=5000):
        for t in chunk["text"].dropna().astype(str):
            if current_id >= start_doc_id:
                yield t, "CSV_2"
            current_id += 1

    # Parquet
    files = glob.glob(os.path.join(PARQUET_DIR, "*.parquet"))

    for file in files:
        df = pd.read_parquet(file, columns=["text"], engine="pyarrow")
        for t in df["text"].dropna().astype(str):
            if current_id >= start_doc_id:
                yield t, os.path.basename(file)
            current_id += 1

In [5]:
def split_text_into_chunks(text, chunk_size=500, overlap=50):
    text = str(text)
    chunks = []
    start = 0

    while start < len(text):
        chunk = text[start:start + chunk_size]
        if len(chunk.strip()) > 50:
            chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [6]:
def normalize(v):
    norm = np.linalg.norm(v, axis=1, keepdims=True)
    norm[norm == 0] = 1
    return v / norm

In [7]:
def load_checkpoint():
    if not os.path.exists(CHECKPOINT_PATH):
        return 0, 0, 0

    with open(CHECKPOINT_PATH, "r") as f:
        doc_id, chunks, batches = f.read().splitlines()

    return int(doc_id), int(chunks), int(batches)

def save_checkpoint(doc_id, chunks, batches):
    with open(CHECKPOINT_PATH, "w") as f:
        f.write(f"{doc_id}\n{chunks}\n{batches}")

In [8]:
dimension = 384

if os.path.exists(FAISS_PATH):
    index = faiss.read_index(FAISS_PATH)
    print("FAISS loaded")
else:
    index = faiss.IndexFlatIP(dimension)
    print("New FAISS index created")

import sqlite3

conn = sqlite3.connect(SQLITE_PATH, timeout=30)

conn.execute("PRAGMA journal_mode=WAL;")
conn.execute("PRAGMA synchronous=NORMAL;")
conn.execute("PRAGMA busy_timeout=30000")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS metadata (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    vector_id INTEGER,
    source TEXT,
    text TEXT
)
""")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_vector_id ON metadata(vector_id)")
conn.commit()

print("New SQLite DB created")

New FAISS index created
New SQLite DB created


In [9]:
last_doc_id, total_chunks, total_batches = load_checkpoint()
print("Resuming from doc_id:", last_doc_id)

if index.ntotal != total_chunks:
    raise Exception(
        f"❌ FAISS-SQLite mismatch! FAISS={index.ntotal}, checkpoint={total_chunks}. STOP."
    )

Resuming from doc_id: 0


In [10]:
print(index.ntotal)
cursor.execute("SELECT COUNT(*) FROM metadata")
print(cursor.fetchone())

0
(0,)


In [ ]:
# import faiss
# import numpy as np

# dimension = 384
# index = faiss.IndexFlatIP(dimension)

# def normalize(v):
#     norm = np.linalg.norm(v, axis=1, keepdims=True)
#     norm[norm == 0] = 1
#     return v / norm

In [ ]:
# import sqlite3
# import faiss
# import numpy as np
# from sentence_transformers import SentenceTransformer
# import torch

# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# conn = sqlite3.connect(SQLITE_PATH)
# cursor = conn.cursor()

# cursor.execute("SELECT text FROM metadata ORDER BY vector_id")
# rows = cursor.fetchall()

# print("Rebuilding FAISS from", len(rows), "rows")

# dimension = 384
# index = faiss.IndexFlatIP(dimension)

# BATCH_SIZE = 64
# batch = []

# for i, (text,) in enumerate(rows):

#     batch.append(text)

#     if len(batch) >= BATCH_SIZE:

#         emb = model.encode(batch, batch_size=64, show_progress_bar=False)
#         emb = np.array(emb).astype("float32")

#         # normalize
#         norm = np.linalg.norm(emb, axis=1, keepdims=True)
#         emb = emb / np.clip(norm, 1e-10, None)

#         index.add(emb)
#         batch = []

#         if i % 10000 == 0:
#             print("Processed:", i)

# # last batch
# if batch:
#     emb = model.encode(batch, batch_size=64, show_progress_bar=False)
#     emb = np.array(emb).astype("float32")

#     norm = np.linalg.norm(emb, axis=1, keepdims=True)
#     emb = emb / np.clip(norm, 1e-10, None)

#     index.add(emb)

# faiss.write_index(index, FAISS_PATH)

# print("✅ FAISS rebuilt:", index.ntotal)

In [11]:
print("FAISS:", index.ntotal)

cursor.execute("SELECT COUNT(*) FROM metadata")
print("SQL:", cursor.fetchone()[0])

print("Checkpoint:", load_checkpoint())

FAISS: 0
SQL: 0
Checkpoint: (0, 0, 0)


In [ ]:
BATCH_SIZE = 64
batch_texts = []
doc_id = last_doc_id

for text, source in stream_all_data(last_doc_id):

    # Chunk text
    chunks = split_text_into_chunks(text)

    for chunk in chunks:

        #Build batch
        batch_texts.append((source, chunk))

        if len(batch_texts) >= BATCH_SIZE:

            texts_only = [x[1] for x in batch_texts]

            #Encode text → embedding
            with torch.no_grad():
                emb = model.encode(texts_only, batch_size=BATCH_SIZE, show_progress_bar=False)

            #Convert + normalize
            emb = np.array(emb).astype("float32")
            emb = normalize(emb)

            #Add to FAISS
            start_id = index.ntotal
            index.add(emb)

            # Each vector gets: unique ID, source, text
            rows = [
                (start_id + i, src, txt)
                for i, (src, txt) in enumerate(batch_texts)
            ]

            # Insert into SQLite
            with conn:
              conn.executemany(
                  "INSERT INTO metadata (vector_id, source, text) VALUES (?, ?, ?)",
                  rows
              )

            cursor.close()
            cursor = conn.cursor()

            conn.commit()


            total_batches += 1

            print(f"Batch {total_batches} | Vectors {index.ntotal}")

            # Save progress
            save_checkpoint(doc_id + 1, index.ntotal, total_batches)
            faiss.write_index(index, FAISS_PATH)

            # Reset batch
            batch_texts = []

    doc_id += 1

Batch 1 | Vectors 64
Batch 2 | Vectors 128
Batch 3 | Vectors 192
Batch 4 | Vectors 256
Batch 5 | Vectors 320
Batch 6 | Vectors 384
Batch 7 | Vectors 448
Batch 8 | Vectors 512
Batch 9 | Vectors 576
Batch 10 | Vectors 640
Batch 11 | Vectors 704
Batch 12 | Vectors 768
Batch 13 | Vectors 832
Batch 14 | Vectors 896
Batch 15 | Vectors 960
Batch 16 | Vectors 1024
Batch 17 | Vectors 1088
Batch 18 | Vectors 1152
Batch 19 | Vectors 1216
Batch 20 | Vectors 1280
Batch 21 | Vectors 1344
Batch 22 | Vectors 1408
Batch 23 | Vectors 1472
Batch 24 | Vectors 1536
Batch 25 | Vectors 1600
Batch 26 | Vectors 1664
Batch 27 | Vectors 1728
Batch 28 | Vectors 1792
Batch 29 | Vectors 1856
Batch 30 | Vectors 1920
Batch 31 | Vectors 1984
Batch 32 | Vectors 2048
Batch 33 | Vectors 2112
Batch 34 | Vectors 2176
Batch 35 | Vectors 2240
Batch 36 | Vectors 2304
Batch 37 | Vectors 2368
Batch 38 | Vectors 2432
Batch 39 | Vectors 2496
Batch 40 | Vectors 2560
Batch 41 | Vectors 2624
Batch 42 | Vectors 2688
Batch 43 | Vector

In [ ]:
if batch_texts:

    texts_only = [x[1] for x in batch_texts]

    with torch.no_grad():
      emb = model.encode(texts_only, batch_size=BATCH_SIZE, show_progress_bar=False)
    emb = np.array(emb).astype("float32")
    emb = normalize(emb)

    start_id = index.ntotal
    index.add(emb)

    rows = [
        (start_id + i, src, txt)
        for i, (src, txt) in enumerate(batch_texts)
    ]

    with conn:
      conn.executemany(
          "INSERT INTO metadata (vector_id, source, text) VALUES (?, ?, ?)",
          rows
      )

    cursor.close()
    cursor = conn.cursor()

    conn.commit()

    total_batches += 1
    next_doc_id = doc_id + 1
    save_checkpoint(next_doc_id, index.ntotal, total_batches)
    faiss.write_index(index, FAISS_PATH)

# ==============================
# 14. CLOSE
# ==============================
conn.close()

print("\nPROCESS COMPLETE")
print("Total vectors (FAISS):", index.ntotal)
print("FAISS vectors:", index.ntotal)
print("Batches:", total_batches)

In [ ]:
# # LOAD
# index = faiss.read_index(FAISS_PATH)

# with open(META_PATH, "rb") as f:
#     metadata = pickle.load(f)